walkthrough for docs

In [ ]:
from pathlib import Path
from metasmith.python_api import *
from metasmith import examples

from local.constants import WORKSPACE_ROOT

In [ ]:
EXAMPLES_DIR = WORKSPACE_ROOT/"src/metasmith/example_resources"
dtypes = DataTypeLibrary.Load(EXAMPLES_DIR/"types/minimal_genomics.yml")

xgdb_path = EXAMPLES_DIR/"data/fosmid.xgdb"
xgdb = DataInstanceLibrary.Load(xgdb_path)
# xgdb = DataInstanceLibrary(xgdb_path)
# xgdb.AddTypeLibrary("genomics", dtypes)
# xgdb.Add([
#     (EXAMPLES_DIR/"fosmid.fna", "./fosmid.fna", "genomics::contigs"),
# ])
# xgdb.PruneTypes()
# xgdb.Save()

refdb_path = EXAMPLES_DIR/"data/references.xgdb"
refdb = DataInstanceLibrary.Load(refdb_path)
# refdb = DataInstanceLibrary(refdb_path)
# refdb.AddTypeLibrary("genomics", dtypes)
# refdb.Add([
#     (EXAMPLES_DIR/"swissprot_bcaa.fna", "./swissprot_bcaa.fna", "genomics::aa_sequences"),
# ])
# refdb.PruneTypes()
# refdb.Save()

trans_path = EXAMPLES_DIR/"transforms/gene_annotation"
transforms = TransformInstanceLibrary.Load(trans_path); # transforms.PruneTypes()

# transforms = TransformInstanceLibrary(trans_path)
# transforms.AddTypeLibrary("genomics", dtypes)
# transforms.AddStub("pprodigal")
# transforms.AddStub("diamond")
# transforms.AddStub("make_diamond_db")
# transforms.Save()

In [ ]:
agent = Agent(
    home = Source.FromLocal(Path("./cache/local_home").resolve()),
    # home = Source.FromLocal((WORKSPACE_ROOT/"docs/source/metasmith_home").resolve()),
)
agent.Deploy()

In [ ]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[transforms],
    targets=[
        dtypes["orf_annotations"].WithLineage([dtypes["contigs"]]),
    ]
)
for step in task.plan.steps:
    print(f">>> {step.transform.name}")
    for x in step.uses:
        print(x.path)
    print("---")
    for x in step.produces:
        print(x.path)
    print()

In [ ]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

In [ ]:
agent.RunWorkflow(task)

In [ ]:
agent.CheckWorkflow(task)